# 11.5 · 情感分析 / Sentiment Analysis

> **课程定位 / Where this fits**
> 第 5 课，**Part 11 · 经典 NLP**。
> Lesson 5, **Part 11 · Classic NLP**.
>
> **情感分析(sentiment analysis)** 判断一段文本的情感倾向(正面/负面)——电商评论、舆情监控、客服质检、股市情绪都靠它。它是文本分类(11.4)的重要专题，但有独特难点：**否定("not good")、程度词、反讽、领域差异**。本课对比两条主线——**词典法**(查情感词典打分，无需训练)和**机器学习法**(TF-IDF+分类器)，并**亲手处理否定**这个经典坑，在真实影评数据上实测。
> **Sentiment analysis** judges the polarity (positive/negative) of text — product reviews, brand monitoring, support QA, market sentiment. A key specialization of text classification (11.4) with unique challenges: **negation ("not good"), intensifiers, sarcasm, domain shift**. We compare two approaches — the **lexicon method** (score with a sentiment dictionary, no training) and **machine learning** (TF-IDF + classifier) — and **handle negation** (the classic pitfall) on real movie-review data.
>
> 💼 **实战/面试视角**："词典法 vs 机器学习 / 否定怎么处理 / 反讽难点 / 跨领域迁移" 是情感/NLP 岗常考。
> 💼 **Practical/interview angle:** "lexicon vs ML / handling negation / sarcasm / cross-domain" — sentiment/NLP questions.

> 📐 **符号约定 / Notation**
> - 极性(polarity) —— 正面(+) / 负面(−) 倾向 / positive/negative orientation
> - 情感词典(lexicon) —— 标好正负的词表 / a list of words tagged positive/negative

> 💡 **面试相关 / Interview-relevant**
> - "词典法 vs 机器学习法的优劣"（出镜率 ★★★★★）
> - "否定/程度词怎么处理"（★★★★★）
> - "情感分析为什么比普通分类难(反讽/领域)"（★★★★）
> - "细粒度/方面级情感(ABSA)"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解情感分析任务与独特难点。
   Understand sentiment analysis and its unique challenges.
2. 实现**词典法**并评估其优劣。
   Implement the lexicon method and assess pros/cons.
3. **处理否定**，看准确率提升。
   Handle negation and see the accuracy gain.
4. 用**机器学习法**对比，理解何时用哪种。
   Compare the ML approach; know when to use which.

## 目录 / TOC
1. [情感分析与难点 ⭐](#1)
2. [词典法（从零）⭐](#2)
3. [处理否定 ⭐](#3)
4. [机器学习法对比 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 情感分析与难点 ⭐ / Sentiment Analysis & Challenges

任务：给一段文本判正面/负面。看似简单，但有不少坑(面试爱问)：
Task: label text positive/negative. Seemingly simple, but full of pitfalls (interview favorites):
- **否定**：`"not good"`、`"isn't bad"` —— 单看词("good"/"bad")会判反。
  **Negation:** `"not good"`, `"isn't bad"` — looking at single words ("good"/"bad") flips the verdict.
- **程度/转折**：`"good but overpriced"` —— 后半句才是重点。
  **Intensity/contrast:** `"good but overpriced"` — the second half dominates.
- **反讽**：`"Oh great, another bug."` —— 字面正面实则负面，极难。
  **Sarcasm:** `"Oh great, another bug."` — literally positive, actually negative; very hard.
- **领域差异**：`"unpredictable"` 对电影是褒义、对刹车是贬义。
  **Domain shift:** `"unpredictable"` is praise for a movie, criticism for brakes.

我们用 **nltk 的 movie_reviews** 语料(2000 篇影评，正负各 1000 篇，已标注)。
We use **nltk's movie_reviews** corpus (2000 reviews, 1000 pos / 1000 neg, labeled).


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, re
import nltk
for r in ["movie_reviews", "opinion_lexicon"]: nltk.download(r, quiet=True)
from nltk.corpus import movie_reviews
sns.set_theme(style="whitegrid")

# 加载影评: 每篇 (原文, 标签) / load reviews: (text, label)
docs = [(movie_reviews.raw(fid), cat)
        for cat in movie_reviews.categories() for fid in movie_reviews.fileids(cat)]
np.random.seed(0); np.random.shuffle(docs)
texts = [d for d, _ in docs]; labels = np.array([1 if c == "pos" else 0 for _, c in docs])  # pos=1, neg=0
print(f"影评数据: {len(texts)} 篇, 正面 {labels.sum()} 篇, 负面 {(labels==0).sum()} 篇")
print("\n一篇负面影评节选:")
print(texts[np.where(labels==0)[0][0]][:200].strip())


<a id="2"></a>
## 2. 词典法（从零）⭐ / Lexicon Method From Scratch

最直接的思路，**完全不用训练**：准备一个**情感词典**(标好哪些词正面、哪些负面)，数一段文本里正面词多还是负面词多。
The most direct idea, **no training at all**: take a **sentiment lexicon** (words tagged positive/negative), and count whether a text has more positive or negative words.

我们用 nltk 的 **opinion_lexicon**(Bing Liu 情感词典，约 2000 正面 + 4800 负面词)。**得分 = 正面词数 − 负面词数**，>0 判正面。
We use nltk's **opinion_lexicon** (Bing Liu's lexicon, ~2000 positive + 4800 negative words). **Score = #positive − #negative**; >0 → positive.

优点：无需标注数据、可解释、即插即用。缺点：忽略否定/词序/语境、领域不适配。
Pros: no labeled data, interpretable, plug-and-play. Cons: ignores negation/order/context, no domain adaptation.


In [ ]:
from nltk.corpus import opinion_lexicon
pos_set = set(opinion_lexicon.positive())                 # 正面词集合 / positive words
neg_set = set(opinion_lexicon.negative())                 # 负面词集合 / negative words
print(f"情感词典: {len(pos_set)} 正面词, {len(neg_set)} 负面词")

def lexicon_score(text):
    toks = re.findall(r"[a-z']+", text.lower())           # 分词 / tokenize
    pos = sum(t in pos_set for t in toks)                 # 正面词计数 / count positive
    neg = sum(t in neg_set for t in toks)                 # 负面词计数 / count negative
    return 1 if pos >= neg else 0                         # 正面词不少于负面词 → 判正面 / predict

pred_lex = np.array([lexicon_score(t) for t in texts])
acc_lex = (pred_lex == labels).mean()
print(f"\n词典法准确率 = {acc_lex:.3f}  (随机=0.5; 无需任何训练数据!)")
print("优点: 零训练/可解释/即用; 缺点: 忽略否定('not good'按词典会判正面)、词序、语境、领域")


<a id="3"></a>
## 3. 处理否定 ⭐ / Handling Negation

词典法最大的漏洞是**否定**：`"not good"` 里有正面词 `good`，词典会误判为正面。经典的轻量修复(面试常考)：**遇到否定词(not/no/never/n't)，把其后若干个词的极性翻转**。
The lexicon's biggest hole is **negation**: `"not good"` contains the positive word `good`, so the lexicon misjudges it positive. A classic lightweight fix (interview favorite): **upon a negation word (not/no/never/n't), flip the polarity of the next few words**.

实现：扫描时若遇到否定词，开启一个"翻转窗口"，窗口内的正面词当负面、负面词当正面来计数。
Implementation: when scanning hits a negation word, open a "flip window"; within it, count positive words as negative and vice versa.


In [ ]:
NEG_WORDS = {"not","no","never","n't","cannot","without","hardly","barely","nor","none"}
def lexicon_score_neg(text, window=3):
    toks = re.findall(r"[a-z']+", text.lower())
    pos = neg = 0; flip = 0                               # flip>0 表示当前处于否定翻转窗口 / inside negation window
    for t in toks:
        if t in NEG_WORDS: flip = window; continue        # 遇否定词 → 开启翻转窗口 / open flip window
        is_pos = t in pos_set; is_neg = t in neg_set
        if flip > 0:                                      # 窗口内极性翻转 / flip polarity inside window
            is_pos, is_neg = is_neg, is_pos
            flip -= 1
        pos += is_pos; neg += is_neg
    return 1 if pos >= neg else 0

pred_neg = np.array([lexicon_score_neg(t) for t in texts])
acc_neg = (pred_neg == labels).mean()
# 在小例子上验证 / sanity-check on tiny examples
for s in ["this movie is good", "this movie is not good", "not bad at all"]:
    print(f"  '{s}' → 无否定处理:{['neg','pos'][lexicon_score(s)]}, 有否定处理:{['neg','pos'][lexicon_score_neg(s)]}")
fig, ax = plt.subplots(figsize=(5.5,3.6))
bars = ax.bar(["词典法", "词典法+否定处理"], [acc_lex, acc_neg], color=["#bbb","#5a9"])
for b,a in zip(bars,[acc_lex,acc_neg]): ax.text(b.get_x()+b.get_width()/2, a+0.005, f"{a:.3f}", ha="center")
ax.set_ylabel("准确率"); ax.set_ylim(0.5,0.8); ax.set_title("否定处理: 修正了玩具例子, 但长文档语料级提升有限")
plt.tight_layout(); plt.show()
print(f"\n词典法 {acc_lex:.3f} → 加否定处理 {acc_neg:.3f}")
print("诚实结论: 否定处理在'not good'这类短句上明显有效(见上面例子);")
print("但在整段长影评上语料级提升很小 —— 长文里情感线索很多, 且粗糙的翻转窗口既修对也改错")
print("启示(面试加分): 玩具例子能修 ≠ 真实指标会涨; 规则法对长文/反讽/领域差异上限明显")


<a id="4"></a>
## 4. 机器学习法对比 + 小结 ⭐ / ML Approach & Summary

**机器学习法**：不靠人工词典，而是从**带标注的数据**里学习哪些词(及组合)指示正/负。用 11.4 的套路：**TF-IDF + 逻辑回归**。它能自动学到领域内的情感词、并通过 bigram 捕捉部分否定("not good")。
**ML approach:** instead of a hand lexicon, **learn from labeled data** which words (and combos) indicate pos/neg. Same recipe as 11.4: **TF-IDF + logistic regression**. It auto-learns domain-specific sentiment words and captures some negation via bigrams ("not good").


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(texts, labels, test_size=0.3, random_state=0, stratify=labels)
pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=3, stop_words=None)),  # 保留停用词(not 重要!) / keep stopwords
                 ("clf", LogisticRegression(max_iter=1000, C=10))])
pipe.fit(Xtr, ytr)
acc_ml = pipe.score(Xte, yte)
print(f"机器学习法(TF-IDF+逻辑回归) test 准确率 = {acc_ml:.3f}")
print(f"对比: 词典法 {acc_lex:.3f}, 词典+否定 {acc_neg:.3f}, 机器学习 {acc_ml:.3f}")
print("注意: 情感分析里 TfidfVectorizer 不去停用词(not/no/very 等承载情感, 与11.2的'BoW常去停用词'相反)\n")

# 看模型学到的最强正/负情感词 / strongest learned pos/neg words
vec = pipe.named_steps["tfidf"]; clf = pipe.named_steps["clf"]
feat = np.array(vec.get_feature_names_out()); coef = clf.coef_[0]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
top_pos = coef.argsort()[-12:]; top_neg = coef.argsort()[:12]
axes[0].barh(feat[top_pos], coef[top_pos], color="#2a9d8f"); axes[0].set_title("最'正面'的词(模型学到)")
axes[1].barh(feat[top_neg], coef[top_neg], color="#e76f51"); axes[1].set_title("最'负面'的词(模型学到)")
plt.tight_layout(); plt.show()
print("机器学习法自动学到领域内情感词(还含 bigram 如 'not funny'), 通常优于通用词典")


```
情感分析: 判正/负极性; 文本分类的专题; 难点=否定/程度/反讽/领域差异
词典法: 数正面词vs负面词, 零训练/可解释/即用; 但忽略否定词序语境 → 上限低
否定处理: 遇 not/no/n't 翻转后续几个词极性; 短句明显有效, 但长文语料级提升有限(粗规则有得有失)
机器学习法: TF-IDF+逻辑回归 从标注数据学情感词; bigram 捕捉部分否定; 通常最好
关键坑: 情感分析'不要去停用词'(not/very 承载情感), 与一般文本分类相反
进阶: 方面级情感(ABSA)/反讽检测; 现代用 BERT 微调(Part 12)效果最好
```

### 💡 面试速查 / Interview cheat-sheet
1. **词典法 vs ML**: 词典零训练/可解释但上限低; ML 学数据、通常更准。
   Lexicon vs ML: lexicon needs no training/interpretable but caps low; ML learns, usually better.
2. **否定处理**: 遇否定词翻转后续词极性(not good→负面)。
   Negation: flip polarity of following words after a negator.
3. **别去停用词**: not/no/very 承载情感(与普通分类相反)。
   Keep stopwords: not/no/very carry sentiment (opposite of generic classification).
4. **难点**: 反讽/领域差异/程度转折, 规则难解。
   Hard cases: sarcasm/domain/contrast, hard for rules.
5. **现代最佳**: 预训练 BERT 微调(Part 12)。
   Modern best: fine-tuned pretrained BERT (Part 12).

### 下一节 / Next
**11.6 主题模型**——前面都是有监督分类。如果有一堆**没有标签**的文档，想自动发现里面有哪些"主题"呢？主题模型(LDA/NMF)无监督地把文档分解成若干主题(词的分布)。我们会可视化发现的主题。
**11.6 Topic Modeling** — so far supervised. Given a pile of **unlabeled** documents, how to auto-discover the "topics" inside? Topic models (LDA/NMF) decompose documents into topics (word distributions) unsupervised. We'll visualize the discovered topics.
